# SPARC Example 08: Hubble Type vs Peak Velocity

**EPS Research RAG Astrophysics Corpus — Unified HI Corpus v7.3**

SPARC includes Hubble morphological type (0=S0 to 11=Irr/BCD).
Galaxy morphology correlates with peak rotation velocity —
earlier types (S0, Sa) rotate faster; late irregulars rotate slowly.

**Important note on corpus fidelity:** The `rotation_curve_corpus_v7.1_flat.csv` and `rotation_curve_corpus_v7.json` are **full-fidelity** — not a summary or veneer. The CSV contains every kinematic parameter published by Lelli et al. (2016) including per-galaxy inclination, distance uncertainties, mass-to-light ratios, and rotation curve statistics. The JSON adds full per-ring data: Vobs, Vgas, Vdisk, Vbul, errV at every radial point. This is the complete published dataset in a single machine-readable file.

**Corpus:** Flynn (2026), Zenodo DOI: 10.5281/zenodo.23442703  
**Source:** Lelli, McGaugh & Schombert (2016), AJ 152, 157  
**Dependencies:** Python 3, numpy, matplotlib, csv (standard library only)

In [ ]:
# ── Colab setup: auto-download corpus from Zenodo ─────────────
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import urllib.request
    CORPORA = {
        'rotation_curve_corpus_v7.json': 'https://zenodo.org/records/23442703/files/rotation_curve_corpus_v7.json',
    }
    for filename, url in CORPORA.items():
        if not os.path.exists(filename):
            print(f"Downloading {filename}...")
            urllib.request.urlretrieve(url, filename)
            print(f"  ✓ {filename}")
        else:
            print(f"  Already present: {filename}")
    print("Ready.")
else:
    print("Running locally — corpus files loaded from working directory.")


In [ ]:
import csv
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import defaultdict

rows = []
with open('rotation_curve_corpus_v7.1_flat.csv') as f:
    for r in csv.DictReader(f):
        try:
            if r['survey']=='SPARC' and r['hubble_type'] and r['vrot_max_kms']:
                rows.append({
                    'type': int(float(r['hubble_type'])),
                    'vmax': float(r['vrot_max_kms']),
                })
        except (ValueError, KeyError):
            pass

TYPE_LABELS = {0:'S0',1:'Sa',2:'Sab',3:'Sb',4:'Sbc',
               5:'Sc',6:'Scd',7:'Sd',8:'Sdm',9:'Sm',10:'Im'}

by_type = defaultdict(list)
for r in rows:
    by_type[r['type']].append(r['vmax'])

types = sorted(by_type.keys())
medians = [np.median(by_type[t]) for t in types]
q25 = [np.percentile(by_type[t], 25) for t in types]
q75 = [np.percentile(by_type[t], 75) for t in types]
labels = [TYPE_LABELS.get(t, str(t)) for t in types]
counts = [len(by_type[t]) for t in types]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Panel 1: Median Vmax by Hubble type
ax = axes[0]
x = np.arange(len(types))
ax.bar(x, medians, color='#1f77b4', alpha=0.8, edgecolor='white')
ax.errorbar(x, medians,
            yerr=[np.array(medians)-np.array(q25),
                  np.array(q75)-np.array(medians)],
            fmt='none', color='black', capsize=4)
ax.set_xticks(x)
ax.set_xticklabels([f'{l}\n(N={c})' for l,c in zip(labels,counts)], fontsize=9)
ax.set_ylabel('Median Vmax (km/s)', fontsize=12)
ax.set_title('Vmax by Hubble Type\nSPARC sample (median ± IQR)', fontsize=11)

# Panel 2: Boxplot
ax2 = axes[1]
ax2.boxplot([by_type[t] for t in types], labels=labels,
            patch_artist=True,
            boxprops=dict(facecolor='#aec7e8', alpha=0.7))
ax2.set_xlabel('Hubble Type', fontsize=12)
ax2.set_ylabel('Vmax (km/s)', fontsize=12)
ax2.set_title('Vmax Distribution by Hubble Type', fontsize=11)

plt.suptitle('HI Corpus v7.3 — Morphology vs Kinematics (SPARC)', fontsize=11)
plt.tight_layout()
plt.savefig('ex08_hubble_type_vmax.png', dpi=150, bbox_inches='tight')
plt.show()
for t, l, m, c in zip(types, labels, medians, counts):
    print(f'{l:5s} (N={c:3d}): median Vmax = {m:.1f} km/s')
